# 1. Cnn

This notebook runs the Intel image classification experiments, saves comparison tables, and checks Keras versus scratch forward propagation for the best shared-parameter CNN.

In [1]:
from pathlib import Path
import sys

def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start] + list(start.parents):
        if (path / 'src').exists() and (path / 'data').exists():
            return path
    raise RuntimeError('Repo root tidak ditemukan. Jalankan dari repo, atau run Setup lebih dulu.')

ROOT = find_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('ROOT:', ROOT)

ROOT: /Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic


In [ ]:
import itertools
import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

from cnn.layers import Conv2D, LocallyConnected2D, MaxPooling2D, AveragePooling2D, Flatten, Dense, Sequential

SEED = 42
IMAGE_SIZE = (150, 150)
CNN_BATCH = 32
CNN_EPOCHS = 5
USE_EXISTING_CNN = True
SKIP_CNN_TRAINING = True
REBUILD_CNN_SCRATCH = False
RUN_FULL_NON_SHARED = False
NON_SHARED_SAMPLE_PER_CLASS = 1

CNN_MODEL_DIR = ROOT / 'models' / 'cnn'
TABLE_DIR = ROOT / 'reports' / 'tables'
FIG_DIR = ROOT / 'reports' / 'figures'
for folder in [CNN_MODEL_DIR, TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = ROOT / 'data/raw/intel/seg_train'
TEST_DIR = ROOT / 'data/raw/intel/seg_test'
CLASSES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASSES)}


In [ ]:
def load_intel_dataset(root, target_size=(150, 150)):
    images = []
    labels = []
    for class_name in CLASSES:
        class_dir = Path(root) / class_name
        for path in sorted(class_dir.glob('*.jpg')):
            img = Image.open(path).convert('RGB').resize(target_size)
            images.append(np.asarray(img, dtype='float32') / 255.0)
            labels.append(CLASS_TO_ID[class_name])
    return np.asarray(images, dtype='float32'), np.asarray(labels, dtype='int64')

X_train_full, y_train_full = load_intel_dataset(TRAIN_DIR, IMAGE_SIZE)
X_test, y_test = load_intel_dataset(TEST_DIR, IMAGE_SIZE)
if len(X_train_full) == 0 or len(X_test) == 0:
    raise RuntimeError('Dataset Intel belum siap. Run Setup lebih dulu.')
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=SEED, stratify=y_train_full
)
print('train:', X_train.shape, 'val:', X_val.shape, 'test:', X_test.shape)

In [ ]:
def make_cnn_model(num_layers, filters, kernel_size, pooling):
    pool_layer = layers.MaxPooling2D if pooling == 'max' else layers.AveragePooling2D
    model_layers = [layers.Input(shape=(*IMAGE_SIZE, 3))]
    for index in range(num_layers):
        model_layers.append(layers.Conv2D(filters, kernel_size, padding='same', activation='relu'))
        if index == 0 or index == num_layers - 1:
            model_layers.append(pool_layer(pool_size=(2, 2)))
    model_layers.extend([layers.Flatten(), layers.Dense(len(CLASSES), activation='softmax')])
    model = models.Sequential(model_layers)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


def normalize_cnn_records(df):
    if df.empty:
        return df
    rename = {
        'experiment_id': 'experiment',
        'pooling_type': 'pooling',
        'keras_weight_path': 'weights_path',
    }
    df = df.rename(columns={old: new for old, new in rename.items() if old in df.columns})
    if 'experiment' not in df.columns and 'index' in df.columns:
        df['experiment'] = df['index'].astype(int) - 1
    for exp_id in range(16):
        mask = df.get('experiment', pd.Series(dtype=int)).astype(int) == exp_id if 'experiment' in df.columns else []
        if len(df) and np.any(mask):
            row_index = df.index[mask][0]
            if 'weights_path' not in df.columns or pd.isna(df.loc[row_index].get('weights_path', np.nan)):
                df.loc[row_index, 'weights_path'] = str(CNN_MODEL_DIR / f'keras_weights_exp_{exp_id}.npy')
            if 'history_path' not in df.columns or pd.isna(df.loc[row_index].get('history_path', np.nan)):
                df.loc[row_index, 'history_path'] = str(CNN_MODEL_DIR / f'history_exp_{exp_id}.pkl')
    return df

configs = []
for num_layers, filters, kernel_size, pooling in itertools.product([2, 3], [32, 64], [3, 5], ['max', 'avg']):
    configs.append({'num_layers': num_layers, 'filters': filters, 'kernel_size': kernel_size, 'pooling': pooling})

record_path = TABLE_DIR / 'cnn_records.csv'
existing_df = normalize_cnn_records(pd.read_csv(record_path)) if record_path.exists() else pd.DataFrame()
records = existing_df.to_dict('records') if not existing_df.empty else []
seen = {int(row['experiment']) for row in records if 'experiment' in row}

def cnn_artifacts_complete():
    if len(seen) < 16:
        return False
    for exp_id in range(16):
        if not (CNN_MODEL_DIR / f'keras_weights_exp_{exp_id}.npy').exists():
            return False
        if not (CNN_MODEL_DIR / f'history_exp_{exp_id}.pkl').exists():
            return False
    return True

if SKIP_CNN_TRAINING and cnn_artifacts_complete():
    print('CNN training skipped; using existing 16 local experiments.')
elif SKIP_CNN_TRAINING:
    raise RuntimeError('CNN artifacts are incomplete but SKIP_CNN_TRAINING=True.')
else:
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train), seed=SEED).batch(CNN_BATCH).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(CNN_BATCH).prefetch(tf.data.AUTOTUNE)
    test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(CNN_BATCH).prefetch(tf.data.AUTOTUNE)
    for exp_id, cfg in enumerate(configs):
        weight_path = CNN_MODEL_DIR / f'keras_weights_exp_{exp_id}.npy'
        history_path = CNN_MODEL_DIR / f'history_exp_{exp_id}.pkl'
        if exp_id in seen and weight_path.exists() and history_path.exists():
            print('skip cnn exp', exp_id)
            continue
        print('training cnn exp', exp_id, cfg)
        tf.keras.backend.clear_session()
        model = make_cnn_model(**cfg)
        t0 = time.time()
        history = model.fit(train_ds, validation_data=val_ds, epochs=CNN_EPOCHS, verbose=1)
        elapsed = time.time() - t0
        preds = np.argmax(model.predict(test_ds, verbose=0), axis=1)[:len(y_test)]
        f1 = f1_score(y_test, preds, average='macro')
        np.save(weight_path, np.array(model.get_weights(), dtype=object), allow_pickle=True)
        with open(history_path, 'wb') as file:
            pickle.dump(history.history, file)
        records.append({
            'experiment': exp_id,
            **cfg,
            'macro_f1': float(f1),
            'runtime_seconds': float(elapsed),
            'weights_path': str(weight_path),
            'history_path': str(history_path),
        })
        pd.DataFrame(records).sort_values('experiment').to_csv(record_path, index=False)

cnn_df = normalize_cnn_records(pd.DataFrame(records)).sort_values('macro_f1', ascending=False)
cnn_df.to_csv(record_path, index=False)
cnn_df.head()


In [ ]:
cnn_df = normalize_cnn_records(pd.read_csv(TABLE_DIR / 'cnn_records.csv'))
best = cnn_df.sort_values('macro_f1', ascending=False).iloc[0]
best_exp = int(best['experiment'])
best_cfg = {
    'num_layers': int(best['num_layers']),
    'filters': int(best['filters']),
    'kernel_size': int(best['kernel_size']),
    'pooling': str(best['pooling']),
}
print('best:', best_exp, best_cfg)

manual_path = TABLE_DIR / 'cnn_manual_comparison.csv'
if USE_EXISTING_CNN and manual_path.exists() and not REBUILD_CNN_SCRATCH:
    manual = pd.read_csv(manual_path)
    print('CNN scratch comparison loaded from existing artifact.')
else:
    best_model = make_cnn_model(**best_cfg)
    best_weights = np.load(CNN_MODEL_DIR / f'keras_weights_exp_{best_exp}.npy', allow_pickle=True)
    best_model.set_weights(list(best_weights))

    sample_idx = []
    per_class = None if RUN_FULL_NON_SHARED else NON_SHARED_SAMPLE_PER_CLASS
    for label in sorted(np.unique(y_test)):
        candidates = np.flatnonzero(y_test == label)
        sample_idx.extend(candidates.tolist() if per_class is None else candidates[:per_class].tolist())
    X_sample = X_test[sample_idx]
    y_sample = y_test[sample_idx]
    keras_pred = np.argmax(best_model.predict(X_sample, verbose=0), axis=1)

    scratch_layers = []
    weight_iter = iter(best_model.get_weights())
    for layer in best_model.layers:
        if isinstance(layer, tf.keras.layers.Conv2D):
            kernel = next(weight_iter); bias = next(weight_iter)
            conv = Conv2D(filters=kernel.shape[-1], kernel_size=kernel.shape[:2], padding=kernel.shape[0] // 2, activation='relu')
            conv.load_keras([kernel, bias])
            scratch_layers.append(conv)
        elif isinstance(layer, tf.keras.layers.MaxPooling2D):
            scratch_layers.append(MaxPooling2D(pool_size=(2, 2), strides=2))
        elif isinstance(layer, tf.keras.layers.AveragePooling2D):
            scratch_layers.append(AveragePooling2D(pool_size=(2, 2), strides=2))
        elif isinstance(layer, tf.keras.layers.Flatten):
            scratch_layers.append(Flatten())
        elif isinstance(layer, tf.keras.layers.Dense):
            kernel = next(weight_iter); bias = next(weight_iter)
            dense = Dense(activation='softmax')
            dense.load_keras([kernel, bias])
            scratch_layers.append(dense)

    t0 = time.time()
    scratch_model = Sequential(scratch_layers)
    scratch_pred = np.argmax(scratch_model.forward(X_sample), axis=1)
    scratch_seconds = time.time() - t0
    shared_params = scratch_model.count_params()

    h, w, channels = IMAGE_SIZE[0], IMAGE_SIZE[1], 3
    non_shared_params = 0
    for layer in best_model.layers:
        if isinstance(layer, tf.keras.layers.Conv2D):
            k_h, k_w = layer.kernel_size
            out_ch = layer.filters
            non_shared_params += h * w * ((k_h * k_w * channels * out_ch) + out_ch)
            channels = out_ch
        elif isinstance(layer, (tf.keras.layers.MaxPooling2D, tf.keras.layers.AveragePooling2D)):
            h //= 2; w //= 2
        elif isinstance(layer, tf.keras.layers.Dense):
            non_shared_params += int(np.prod(layer.kernel.shape) + np.prod(layer.bias.shape))

    manual = pd.DataFrame([
        {'implementation': 'keras_shared', 'macro_f1': f1_score(y_sample, keras_pred, average='macro'), 'runtime_seconds': np.nan, 'params': int(best_model.count_params()), 'numerical_match': True, 'shared_parameters': True},
        {'implementation': 'scratch_numpy_shared', 'macro_f1': f1_score(y_sample, scratch_pred, average='macro'), 'runtime_seconds': scratch_seconds, 'params': int(shared_params), 'numerical_match': bool(np.array_equal(keras_pred, scratch_pred)), 'shared_parameters': True},
        {'implementation': 'scratch_numpy_non_shared', 'macro_f1': np.nan, 'runtime_seconds': np.nan, 'params': int(non_shared_params), 'numerical_match': False, 'shared_parameters': False, 'macro_f1_small_batch': np.nan, 'parameter_explosion': float(non_shared_params / max(shared_params, 1))},
    ])
    manual.to_csv(manual_path, index=False)

manual


In [ ]:
cnn_df = normalize_cnn_records(pd.read_csv(TABLE_DIR / 'cnn_records.csv'))
for group_col, out_name in [
    ('num_layers', 'cnn_by_num_layers.csv'),
    ('filters', 'cnn_by_filters.csv'),
    ('kernel_size', 'cnn_by_kernel_size.csv'),
    ('pooling', 'cnn_by_pooling_type.csv'),
]:
    cnn_df.groupby(group_col)[['macro_f1']].mean().reset_index().to_csv(TABLE_DIR / out_name, index=False)

plt.figure(figsize=(10, 4))
plt.bar(cnn_df['experiment'].astype(str), cnn_df['macro_f1'])
plt.xlabel('Experiment')
plt.ylabel('Macro F1')
plt.title('CNN Macro F1')
plt.tight_layout()
plt.savefig(FIG_DIR / 'cnn_macro_f1.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for axis, (group_col, title) in zip(axes.ravel(), [('num_layers', 'Conv Layers'), ('filters', 'Filters'), ('kernel_size', 'Kernel Size'), ('pooling', 'Pooling')]):
    summary = cnn_df.groupby(group_col)['macro_f1'].mean().reset_index()
    axis.bar(summary[group_col].astype(str), summary['macro_f1'])
    axis.set_title(title)
    axis.set_ylabel('Macro F1')
fig.tight_layout()
fig.savefig(FIG_DIR / 'cnn_hyperparameters.png', dpi=150)
plt.show()

loss_rows = []
for row in cnn_df.itertuples():
    history_path = Path(getattr(row, 'history_path'))
    if not history_path.is_absolute():
        history_path = ROOT / history_path
    if not history_path.exists():
        continue
    with open(history_path, 'rb') as file:
        history = pickle.load(file)
    for epoch, (loss, val_loss) in enumerate(zip(history.get('loss', []), history.get('val_loss', [])), start=1):
        loss_rows.append({'experiment': int(row.experiment), 'epoch': epoch, 'loss': loss, 'val_loss': val_loss})
loss_df = pd.DataFrame(loss_rows)
if not loss_df.empty:
    plt.figure(figsize=(10, 5))
    for exp_id, part in loss_df.groupby('experiment'):
        plt.plot(part['epoch'], part['val_loss'], alpha=0.45, label=str(exp_id))
    plt.title('CNN Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'cnn_validation_loss.png', dpi=150)
    plt.show()

manual = pd.read_csv(TABLE_DIR / 'cnn_manual_comparison.csv')
if {'implementation', 'params'}.issubset(manual.columns):
    plt.figure(figsize=(8, 4))
    plt.bar(manual['implementation'], pd.to_numeric(manual['params'], errors='coerce'))
    plt.yscale('log')
    plt.xticks(rotation=20, ha='right')
    plt.ylabel('Parameters (log scale)')
    plt.title('Shared vs Non-shared Parameters')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'cnn_shared_parameters.png', dpi=150)
    plt.show()
